# Partie E — Fine-tuning : mono-tâche vs multi-tâche

Dans cette partie, nous évaluons l'apport du **fine-tuning** de deux modèles de langue pré-entraînés (PLMs) spécialisés sur des tweets arabes :
1. `aubmindlab/bert-base-arabertv02-twitter`
2. `UBC-NLP/MARBERTv2`

### Objectifs :
- **E1. Mono-tâche (sarcasme)** : encodeur + tête linéaire, perte pondérée (weighted cross-entropy) pour contrer le déséquilibre de classe (~17 % sarcastique), seuil de décision optimisé sur `val`.
- **E2. Multi-tâche (sarcasme + sentiment)** : encodeur partagé + 2 têtes (sarcasme à 2 classes, sentiment à 3 classes `NEG/NEU/POS`).
  $$\mathcal{L} = \mathcal{L}_{sarc} + \lambda \cdot \mathcal{L}_{sent}, \quad \lambda \in \{0.3, 0.5, 1.0\}$$
  *Optionnel :* 3e tête « dialecte » (5 classes).
- **Protocole d'entraînement** :
  - Learning rate : `2e-5`, Optimizer : `AdamW` (weight decay = `0.01`)
  - Batch size : `32`, Époques : 3 à 5, Warmup linéaire : 10 %
  - Mixed Precision : `fp16`
  - Early Stopping : surveille le **F1-sarcastique sur val** (patience = 2)
  - Évaluation finale sur 3 graines (`seed = [42, 1337, 2024]`) avec moyenne ± écart-type.

In [1]:
import sys
from pathlib import Path

# Add src to Python path
SRC_PATH = Path("../src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import torch
import pandas as pd
import numpy as np

from data import load_and_split
from models.multitask import MonoTask, MultiTask
from train.plm_trainer import TrainConfig, train_plm, compare_lambdas, run_seeds

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
Device: NVIDIA GeForce GTX 1660 Ti


## 1. Chargement des données et vérification des distributions

In [2]:
# Chargement officiel avec suppression des fuites de test et doublons contradictoires
splits = load_and_split()
train_df = splits["train"]
val_df = splits["val"]

print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print("\nDistribution du sarcasme dans train :", train_df['sarcasm'].value_counts().to_dict())
print("Distribution du sentiment dans train :", train_df['sentiment'].value_counts().to_dict())
print("Distribution des dialectes dans train:", train_df['dialect'].value_counts().to_dict())

Train size : 11195
Val size   : 1255

Distribution du sarcasme dans train : {False: 9294, True: 1901}
Distribution du sentiment dans train : {'NEU': 5131, 'NEG': 4097, 'POS': 1967}
Distribution des dialectes dans train: {'msa': 7654, 'egypt': 2385, 'levant': 565, 'gulf': 554, 'magreb': 37}


## 2. Architecture des modèles : Mono-tâche vs Multi-tâche

- **Mono-tâche** : Encodeur PLM + Tête linéaire de classification sarcasme (2 classes).
- **Multi-tâche** : Encodeur partagé (hard parameter sharing) extrayant la représentation `[CLS]` (dimension $h=768$), puis 2 projections linéaires distinctes vers les logits de sarcasme et de sentiment.

In [3]:
# Inspection de l'architecture MultiTask
test_model = MultiTask("aubmindlab/bert-base-arabertv02-twitter", use_dialect=False)
n_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Modèle Multi-tâche initialisé avec {n_params:,} paramètres entraînables.")
print(test_model)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modèle Multi-tâche initialisé avec 135,197,189 paramètres entraînables.
MultiTask(
  (enc): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(64000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
    

## 3. E1 — Mono-tâche (Sarcasme)

Entraînement avec :
- **Perte pondérée** : $w_c = \frac{N}{2 N_c}$ pour pénaliser les erreurs sur la classe minoritaire sarcastique.
- **Optimisation du seuil de décision sur val** : balayage de $\tau \in [0.01, 0.99]$ pour maximiser le F1-sarcastique.
- **Early stopping** : surveillance du F1-sarcastique sur `val` (patience = 2).

In [ ]:
# E1.1: AraBERTv02-Twitter (Mono-tâche)
cfg_arabert_mono = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="mono",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

print("=== Entraînement E1 : AraBERTv02-Twitter (Mono-tâche) ===")
model_arabert_mono, metrics_arabert_mono = train_plm(train_df, val_df, cfg_arabert_mono)

=== Entraînement E1 : AraBERTv02-Twitter (Mono-tâche) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 | Train Loss: 0.4993 | Val F1-Sarc: 0.6653 (at thr=0.71) | Val PR-AUC: 0.6966 | Val Macro-F1: 0.7933
  -> New best validation F1 checkpoint saved: 0.6653
Epoch 2/4 | Train Loss: 0.3291 | Val F1-Sarc: 0.6809 (at thr=0.72) | Val PR-AUC: 0.7050 | Val Macro-F1: 0.7994
  -> New best validation F1 checkpoint saved: 0.6809
Epoch 3/4 | Train Loss: 0.2304 | Val F1-Sarc: 0.6780 (at thr=0.67) | Val PR-AUC: 0.6933 | Val Macro-F1: 0.8017


In [ ]:
# E1.2: MARBERTv2 (Mono-tâche)
cfg_marbert_mono = TrainConfig(
    model_name="UBC-NLP/MARBERTv2",
    task_mode="mono",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

print("=== Entraînement E1 : MARBERTv2 (Mono-tâche) ===")
model_marbert_mono, metrics_marbert_mono = train_plm(train_df, val_df, cfg_marbert_mono)

## 4. E2 — Multi-tâche (Sarcasme + Sentiment)

Nous explorons l'impact du coefficient de tâche $\lambda \in \{0.3, 0.5, 1.0\}$ sur `val` pour trouver le meilleur compromis :
$$\mathcal{L} = \mathcal{L}_{sarc} + \lambda \cdot \mathcal{L}_{sent}$$

In [ ]:
# Grille de recherche pour lambda sur le meilleur PLM (ou les deux)
cfg_multi_base = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="multi",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

results_lambdas = compare_lambdas(train_df, val_df, cfg_multi_base, lambdas=[0.3, 0.5, 1.0])
results_lambdas

## 5. Évaluation multi-graines finale (3 graines : 42, 1337, 2024)

Pour le modèle final retenu (par exemple Multi-tâche avec le meilleur $\lambda$), nous exécutons 3 entraînements indépendants avec des graines aléatoires distinctes pour rapporter la moyenne $\pm$ écart-type, conformément aux exigences scientifiques du TP.

In [ ]:
# Configuration retenue pour les 3 graines
best_lambda = results_lambdas.loc[results_lambdas['val_f1_sarcasm'].idxmax()]['lambda']
print(f"Meilleur lambda retenu sur validation : {best_lambda}")

cfg_final = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="multi",
    lam_sent=float(best_lambda),
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
)

df_runs, summary = run_seeds(train_df, val_df, cfg_final, seeds=[42, 1337, 2024])

## 6. Synthèse des Résultats Partie E

Comparez ici :
1. **Mono-tâche vs Multi-tâche** : est-ce que prédire conjointement le sentiment aide à détecter le sarcasme ? (Lien théorique : le sarcasme exprime souvent un sentiment incongru / négatif masqué par des mots positifs).
2. **Impact du seuil $\tau^*$** : comment varie le seuil optimal par rapport au 0.5 par défaut ?
3. **AraBERT vs MARBERT** : comparaison des représentations tweet.